# 02. Exploratory Data Analysis (EDA) — Fraud Sentinel

This notebook performs comprehensive exploratory data analysis on the preprocessed **FraudSentinel Banking Dataset** using Pandas, NumPy, and Matplotlib.

### Analysis Sections:
1. Fraud vs Legitimate Class Distribution & Imbalance Ratio
2. Transaction Amount Distributions (Raw & Log-scaled)
3. Fraud Rates across Categorical Attributes (Transaction Type, Device Type, Location)
4. Temporal Trends (Hour of Day, Day of Week, Nighttime, Weekend)
5. Customer Behavior & Velocity Features
6. Key Correlation Drivers with Target `is_fraud`

In [ ]:
# Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Styling
plt.style.use('default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

train_df = pd.read_csv('../data/processed/fraudsentinel_train.csv')
print(f'Train Dataset Shape: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')

## 1. Class Imbalance Analysis

In [ ]:
counts = train_df['is_fraud'].value_counts()
pcts = train_df['is_fraud'].value_counts(normalize=True) * 100

print(f'Legitimate (0): {counts[0]:,} ({pcts[0]:.2f}%)')
print(f'Fraudulent (1): {counts[1]:,} ({pcts[1]:.2f}%)')
print(f'Imbalance Ratio: {counts[0]/counts[1]:.2f} : 1')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Legitimate (0)', 'Fraudulent (1)'], counts.values, color=['#1f77b4', '#d62728'], width=0.45)
ax.set_title('Class Distribution (Training Fold)', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 100, f'{yval:,}\n({yval/len(train_df)*100:.2f}%)', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Transaction Amount Distribution & Scaling

In [ ]:
print(train_df.groupby('is_fraud')['transaction_amount'].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(train_df[train_df['is_fraud'] == 0]['transaction_amount'], bins=30, alpha=0.6, color='#1f77b4', label='Legitimate', density=True)
ax1.hist(train_df[train_df['is_fraud'] == 1]['transaction_amount'], bins=30, alpha=0.6, color='#d62728', label='Fraudulent', density=True)
ax1.set_title('Raw Transaction Amount Density')
ax1.set_xlabel('Amount ($)')
ax1.legend()

ax2.hist(train_df[train_df['is_fraud'] == 0]['amount_log'], bins=30, alpha=0.6, color='#1f77b4', label='Legitimate', density=True)
ax2.hist(train_df[train_df['is_fraud'] == 1]['amount_log'], bins=30, alpha=0.6, color='#d62728', label='Fraudulent', density=True)
ax2.set_title('Log-Transformed Amount Density')
ax2.set_xlabel('log1p(Amount)')
ax2.legend()
plt.tight_layout()
plt.show()

## 3. Categorical Risk Factors

In [ ]:
dev_cols = [c for c in train_df.columns if c.startswith('device_type=')]
dev_rates = {c.split('=')[1]: train_df[train_df[c] == 1]['is_fraud'].mean() * 100 for c in dev_cols}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dev_rates.keys(), dev_rates.values(), color='#ff7f0e', width=0.45)
ax.set_title('Fraud Rate (%) by Device Type', fontweight='bold')
ax.set_ylabel('Fraud Percentage (%)')
plt.tight_layout()
plt.show()

## 4. Key Feature Correlations with `is_fraud`

In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.drop(['transaction_year'], errors='ignore')
corrs = train_df[num_cols].corr()['is_fraud'].drop('is_fraud').sort_values()

print('--- TOP POSITIVE CORRELATIONS WITH IS_FRAUD ---')
print(corrs.tail(10))
print('\n--- TOP NEGATIVE CORRELATIONS WITH IS_FRAUD ---')
print(corrs.head(10))

plt.figure(figsize=(10, 6))
top_corrs = pd.concat([corrs.head(5), corrs.tail(5)])
plt.barh(top_corrs.index, top_corrs.values, color=['#d62728' if v > 0 else '#1f77b4' for v in top_corrs.values])
plt.title('Top 10 Feature Correlations with is_fraud', fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()